
## File 02 — Databricks Workflow Orchestration:


This notebook coordinates the execution of your E-Commerce Lakehouse pipeline.

The execution sequence will be:

- Bronze incremental ingestion.
- Silver incremental processing.
- Data quality validation.
- Business rule validation.
- Referential integrity validation.
- Orphan record handling.
- Gold analytics refresh.
- Final incremental pipeline validation.

**Important design decision: **

- We'll use Folder 09 File 02 as the primary incremental Silver processing path. 
- Do not run Folder 09 File 03 as a competing Silver writer in the same workflow.

In [0]:
# FEATURES:

# - Notebook path configuration
# - Job parameters
# - Layer-based task selection
# - Dependency checks
# - Failure and skip handling
# - Delta execution reporting
# - Existing report-table schema compatibility


# ============================================================
# 1. IMPORTS
# ============================================================

from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType
)

import traceback
import uuid


# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

CATALOG = "workspace"

GOLD_SCHEMA = "ecommerce_gold"

REPORT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.workflow_execution_report"
)

WORKSPACE_ROOT = (
    "/Workspace/Data-Enginnering/"
    "Data-Engineering-Python/"
    "E-Commerce-Lakehouse-Project"
)

TIMEOUT_SECONDS = 0


# ============================================================
# 3. NOTEBOOK PATH CONFIGURATION
# ============================================================
# These are the confirmed Workspace notebook paths.
# Do not add ".ipynb" to dbutils.notebook.run() paths.
# ============================================================

NOTEBOOK_PATHS = {

    "bronze_ingestion": (
        f"{WORKSPACE_ROOT}/09_incremental_ingestion/"
        "01_incremental_bronze_ingestion"
    ),

    "silver_processing": (
        f"{WORKSPACE_ROOT}/09_incremental_ingestion/"
        "02_incremental_silver_processing"
    ),

    "delta_merge_audit": (
        f"{WORKSPACE_ROOT}/09_incremental_ingestion/"
        "03_incremental_delta_merge"
    ),

    "gold_refresh": (
        f"{WORKSPACE_ROOT}/09_incremental_ingestion/"
        "04_incremental_gold_refresh"
    ),

    "pipeline_validation": (
        f"{WORKSPACE_ROOT}/09_incremental_ingestion/"
        "05_incremental_pipeline_validation"
    ),

    "data_quality": (
        f"{WORKSPACE_ROOT}/05_data_quality_and_quarantine/"
        "01_reusable_data_quality_framework"
    ),

    "business_rule_validation": (
        f"{WORKSPACE_ROOT}/05_data_quality_and_quarantine/"
        "04_business_rule_validation"
    ),

    "referential_integrity": (
        f"{WORKSPACE_ROOT}/06_referential_integrity_validation/"
        "01_referential_integrity_validation"
    ),

    "orphan_handling": (
        f"{WORKSPACE_ROOT}/06_referential_integrity_validation/"
        "02_orphan_record_handling"
    ),

    "referential_integrity_dashboard": (
        f"{WORKSPACE_ROOT}/06_referential_integrity_validation/"
        "04_referential_integrity_dashboard"
    )
}


# ============================================================
# 4. NOTEBOOK PARAMETERS
# ============================================================

def create_widget_if_missing(name, default_value):

    try:
        dbutils.widgets.get(name)

    except Exception:
        dbutils.widgets.text(name, default_value)


create_widget_if_missing(
    "pipeline_name",
    "ecommerce_lakehouse_pipeline"
)

create_widget_if_missing(
    "dataset_name",
    "all"
)

create_widget_if_missing(
    "pipeline_layer",
    "ALL"
)

create_widget_if_missing(
    "execution_mode",
    "INCREMENTAL"
)

create_widget_if_missing(
    "stop_on_failure",
    "true"
)

create_widget_if_missing(
    "run_id",
    ""
)


# ============================================================
# 5. READ PARAMETERS
# ============================================================

PIPELINE_NAME = (
    dbutils.widgets.get("pipeline_name").strip()
)

DATASET_NAME = (
    dbutils.widgets.get("dataset_name").strip().lower()
)

PIPELINE_LAYER = (
    dbutils.widgets.get("pipeline_layer").strip().upper()
)

EXECUTION_MODE = (
    dbutils.widgets.get("execution_mode").strip().upper()
)

STOP_ON_FAILURE = (
    dbutils.widgets.get("stop_on_failure")
    .strip()
    .lower()
    in ("true", "1", "yes", "y")
)

RUN_ID = (
    dbutils.widgets.get("run_id").strip()
)

if not RUN_ID:
    RUN_ID = str(uuid.uuid4())


# ============================================================
# 6. PARAMETER VALIDATION
# ============================================================

VALID_LAYERS = {
    "ALL",
    "BRONZE",
    "SILVER",
    "QUALITY",
    "REFERENTIAL_INTEGRITY",
    "GOLD",
    "VALIDATION_ONLY"
}

VALID_EXECUTION_MODES = {
    "INCREMENTAL",
    "FULL_REFRESH"
}


if not PIPELINE_NAME:
    raise ValueError(
        "pipeline_name cannot be empty."
    )

if not DATASET_NAME:
    raise ValueError(
        "dataset_name cannot be empty."
    )

if PIPELINE_LAYER not in VALID_LAYERS:
    raise ValueError(
        f"Invalid pipeline_layer: {PIPELINE_LAYER}. "
        f"Allowed values: {sorted(VALID_LAYERS)}"
    )

if EXECUTION_MODE not in VALID_EXECUTION_MODES:
    raise ValueError(
        f"Invalid execution_mode: {EXECUTION_MODE}. "
        f"Allowed values: {sorted(VALID_EXECUTION_MODES)}"
    )


# ============================================================
# 7. WORKFLOW TASK DEFINITIONS
# ============================================================
# delta_merge_audit is an audit task.
# Its current implementation is NOT a true incremental MERGE.
# ============================================================

WORKFLOW_TASKS = [

    {
        "task_name": "bronze_incremental_ingestion",
        "path_key": "bronze_ingestion",
        "layer": "BRONZE",
        "depends_on": []
    },

    {
        "task_name": "silver_incremental_processing",
        "path_key": "silver_processing",
        "layer": "SILVER",
        "depends_on": [
            "bronze_incremental_ingestion"
        ]
    },

    {
        "task_name": "incremental_delta_merge_audit",
        "path_key": "delta_merge_audit",
        "layer": "SILVER",
        "depends_on": [
            "silver_incremental_processing"
        ]
    },

    {
        "task_name": "data_quality_validation",
        "path_key": "data_quality",
        "layer": "QUALITY",
        "depends_on": [
            "silver_incremental_processing"
        ]
    },

    {
        "task_name": "business_rule_validation",
        "path_key": "business_rule_validation",
        "layer": "QUALITY",
        "depends_on": [
            "data_quality_validation"
        ]
    },

    {
        "task_name": "referential_integrity_validation",
        "path_key": "referential_integrity",
        "layer": "REFERENTIAL_INTEGRITY",
        "depends_on": [
            "business_rule_validation"
        ]
    },

    {
        "task_name": "orphan_record_handling",
        "path_key": "orphan_handling",
        "layer": "REFERENTIAL_INTEGRITY",
        "depends_on": [
            "referential_integrity_validation"
        ]
    },

    {
        "task_name": "referential_integrity_dashboard",
        "path_key": "referential_integrity_dashboard",
        "layer": "REFERENTIAL_INTEGRITY",
        "depends_on": [
            "orphan_record_handling"
        ]
    },

    {
        "task_name": "gold_analytics_refresh",
        "path_key": "gold_refresh",
        "layer": "GOLD",
        "depends_on": [
            "referential_integrity_dashboard"
        ]
    },

    {
        "task_name": "final_pipeline_validation",
        "path_key": "pipeline_validation",
        "layer": "VALIDATION",
        "depends_on": [
            "gold_analytics_refresh"
        ]
    }
]


# ============================================================
# 8. SELECT TASKS BASED ON PIPELINE LAYER
# ============================================================

LAYER_TASKS = {

    "ALL": {
        "BRONZE",
        "SILVER",
        "QUALITY",
        "REFERENTIAL_INTEGRITY",
        "GOLD",
        "VALIDATION"
    },

    "BRONZE": {
        "BRONZE"
    },

    "SILVER": {
        "SILVER"
    },

    "QUALITY": {
        "QUALITY"
    },

    "REFERENTIAL_INTEGRITY": {
        "REFERENTIAL_INTEGRITY"
    },

    "GOLD": {
        "GOLD"
    },

    "VALIDATION_ONLY": {
        "VALIDATION"
    }
}


ALLOWED_TASK_LAYERS = LAYER_TASKS[PIPELINE_LAYER]


SELECTED_TASKS = [
    task
    for task in WORKFLOW_TASKS
    if task["layer"] in ALLOWED_TASK_LAYERS
]


if not SELECTED_TASKS:
    raise ValueError(
        f"No workflow tasks selected for layer: {PIPELINE_LAYER}"
    )


# ============================================================
# 9. WORKFLOW CONFIGURATION SUMMARY
# ============================================================

print("=" * 90)
print("E-COMMERCE LAKEHOUSE - WORKFLOW CONFIGURATION")
print("=" * 90)

print(f"Pipeline name       : {PIPELINE_NAME}")
print(f"Run ID              : {RUN_ID}")
print(f"Dataset             : {DATASET_NAME}")
print(f"Pipeline layer      : {PIPELINE_LAYER}")
print(f"Execution mode      : {EXECUTION_MODE}")
print(f"Stop on failure     : {STOP_ON_FAILURE}")
print(f"Configured tasks    : {len(WORKFLOW_TASKS)}")
print(f"Selected tasks      : {len(SELECTED_TASKS)}")

print("=" * 90)
print("SELECTED WORKFLOW TASKS")
print("=" * 90)

for index, task in enumerate(SELECTED_TASKS, start=1):

    print(
        f"{index}. {task['task_name']} "
        f"({task['layer']})"
    )

    print(
        f"   Path: {NOTEBOOK_PATHS[task['path_key']]}"
    )


# ============================================================
# 10. CANONICAL EXECUTION REPORT SCHEMA
# ============================================================
# IMPORTANT:
# This matches the existing Delta table schema:
#
# execution_id       STRING
# run_id             STRING
# pipeline_name      STRING
# task_name          STRING
# pipeline_layer     STRING
# notebook_path      STRING
# execution_status   STRING
# start_timestamp    TIMESTAMP
# end_timestamp      TIMESTAMP
# duration_seconds   DOUBLE
# error_message      STRING
# notebook_output    STRING
#
# Do not append a different schema to this table.
# ============================================================

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"
)


REPORT_SCHEMA = StructType([

    StructField(
        "execution_id",
        StringType(),
        False
    ),

    StructField(
        "run_id",
        StringType(),
        True
    ),

    StructField(
        "pipeline_name",
        StringType(),
        True
    ),

    StructField(
        "task_name",
        StringType(),
        True
    ),

    StructField(
        "pipeline_layer",
        StringType(),
        True
    ),

    StructField(
        "notebook_path",
        StringType(),
        True
    ),

    StructField(
        "execution_status",
        StringType(),
        True
    ),

    StructField(
        "start_timestamp",
        TimestampType(),
        True
    ),

    StructField(
        "end_timestamp",
        TimestampType(),
        True
    ),

    StructField(
        "duration_seconds",
        DoubleType(),
        True
    ),

    StructField(
        "error_message",
        StringType(),
        True
    ),

    StructField(
        "notebook_output",
        StringType(),
        True
    )
])


# ============================================================
# 11. CREATE OR VALIDATE REPORT TABLE
# ============================================================

if not spark.catalog.tableExists(REPORT_TABLE):

    empty_report_df = spark.createDataFrame(
        [],
        REPORT_SCHEMA
    )

    (
        empty_report_df.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(REPORT_TABLE)
    )

    print(
        f"Created execution report table: {REPORT_TABLE}"
    )

else:

    existing_schema = spark.table(
        REPORT_TABLE
    ).schema

    expected_schema = REPORT_SCHEMA

    existing_columns = [
        (field.name, field.dataType.simpleString())
        for field in existing_schema.fields
    ]

    expected_columns = [
        (field.name, field.dataType.simpleString())
        for field in expected_schema.fields
    ]

    if existing_columns != expected_columns:

        print("\nREPORT TABLE SCHEMA MISMATCH")
        print("Existing schema:")
        print(existing_columns)

        print("\nExpected schema:")
        print(expected_columns)

        raise RuntimeError(
            "The existing workflow_execution_report table "
            "does not match the canonical 12-column schema. "
            "No workflow tasks were started. Align the table "
            "and every notebook that writes to it before retrying."
        )

    print(
        f"Existing execution report table validated: {REPORT_TABLE}"
    )


# ============================================================
# 12. REPORT ROW COLLECTION
# ============================================================

report_rows = []

task_status = {}


def utc_now():
    """
    Return a timezone-naive UTC datetime for Spark TimestampType.
    All report timestamps are stored as UTC.
    """

    return datetime.now(timezone.utc).replace(
        tzinfo=None
    )


workflow_start_time = utc_now()


# ============================================================
# 13. ADD REPORT ROW
# ============================================================

def add_report_row(
    task_name,
    notebook_path,
    task_layer,
    status,
    start_time,
    end_time,
    duration_seconds,
    error_message=None,
    notebook_output=None
):

    report_rows.append({

        "execution_id": str(uuid.uuid4()),

        "run_id": RUN_ID,

        "pipeline_name": PIPELINE_NAME,

        "task_name": task_name,

        "pipeline_layer": task_layer,

        "notebook_path": notebook_path,

        "execution_status": status,

        "start_timestamp": start_time,

        "end_timestamp": end_time,

        "duration_seconds": float(
            duration_seconds
        ),

        "error_message": (
            str(error_message)[:12000]
            if error_message is not None
            else None
        ),

        "notebook_output": (
            str(notebook_output)[:4000]
            if notebook_output is not None
            else None
        )
    })


# ============================================================
# 14. CHILD NOTEBOOK PARAMETERS
# ============================================================

def build_task_parameters():

    return {

        "pipeline_name": PIPELINE_NAME,

        "run_id": RUN_ID,

        "dataset_name": DATASET_NAME,

        "pipeline_layer": PIPELINE_LAYER,

        "execution_mode": EXECUTION_MODE
    }


# ============================================================
# 15. DEPENDENCY VALIDATION
# ============================================================

def check_task_dependencies(task):
    """
    If a dependency is part of this execution,
    it must have succeeded.

    If a dependency is not selected in this execution,
    it is treated as an already-completed upstream dependency.

    Example:
    Running only GOLD assumes upstream Silver and
    Referential Integrity outputs already exist.
    """

    unsuccessful_dependencies = []

    for dependency in task["depends_on"]:

        if dependency in task_status:

            if task_status[dependency] != "SUCCESS":

                unsuccessful_dependencies.append(
                    dependency
                )

    return unsuccessful_dependencies


# ============================================================
# 16. SKIPPED TASK HANDLING
# ============================================================

def mark_task_skipped(
    task,
    reason
):

    task_name = task["task_name"]

    notebook_path = NOTEBOOK_PATHS[
        task["path_key"]
    ]

    task_layer = task["layer"]

    now = utc_now()

    task_status[task_name] = "SKIPPED"

    print("\n" + "-" * 80)
    print(f"TASK SKIPPED: {task_name}")
    print(f"Reason: {reason}")
    print("-" * 80)

    add_report_row(

        task_name=task_name,

        notebook_path=notebook_path,

        task_layer=task_layer,

        status="SKIPPED",

        start_time=now,

        end_time=now,

        duration_seconds=0.0,

        error_message=reason,

        notebook_output=None
    )


# ============================================================
# 17. TASK EXECUTION
# ============================================================

def execute_workflow_task(task):

    task_name = task["task_name"]

    path_key = task["path_key"]

    task_layer = task["layer"]

    notebook_path = NOTEBOOK_PATHS[path_key]


    print("\n" + "=" * 90)
    print(f"STARTING TASK: {task_name}")
    print(f"Layer         : {task_layer}")
    print(f"Notebook      : {notebook_path}")
    print("=" * 90)


    # --------------------------------------------------------
    # 17.1 CHECK DEPENDENCIES
    # --------------------------------------------------------

    dependency_failures = check_task_dependencies(
        task
    )

    if dependency_failures:

        reason = (
            "Unsuccessful dependencies: "
            + ", ".join(dependency_failures)
        )

        mark_task_skipped(
            task,
            reason
        )

        return "SKIPPED"


    # --------------------------------------------------------
    # 17.2 START TASK
    # --------------------------------------------------------

    task_start = utc_now()

    print(
        f"Task started at UTC: {task_start.isoformat()}"
    )


    # --------------------------------------------------------
    # 17.3 EXECUTE CHILD NOTEBOOK
    # --------------------------------------------------------

    try:

        notebook_output = dbutils.notebook.run(

            notebook_path,

            TIMEOUT_SECONDS,

            build_task_parameters()
        )


        task_end = utc_now()

        duration = (
            task_end - task_start
        ).total_seconds()


        task_status[task_name] = "SUCCESS"


        print("\nTASK SUCCESS")
        print(f"Task name : {task_name}")
        print(f"Duration  : {duration:.2f} seconds")


        if notebook_output:

            print(
                f"Notebook output: {notebook_output}"
            )


        add_report_row(

            task_name=task_name,

            notebook_path=notebook_path,

            task_layer=task_layer,

            status="SUCCESS",

            start_time=task_start,

            end_time=task_end,

            duration_seconds=duration,

            error_message=None,

            notebook_output=notebook_output
        )


        return "SUCCESS"


    # --------------------------------------------------------
    # 17.4 FAILURE HANDLING
    # --------------------------------------------------------

    except Exception as exc:

        task_end = utc_now()

        duration = (
            task_end - task_start
        ).total_seconds()


        error_text = (
            f"{type(exc).__name__}: {str(exc)}"
        )

        full_traceback = traceback.format_exc()


        task_status[task_name] = "FAILED"


        print("\nTASK FAILED")
        print(f"Task name: {task_name}")
        print(f"Error: {error_text}")
        print(full_traceback)


        add_report_row(

            task_name=task_name,

            notebook_path=notebook_path,

            task_layer=task_layer,

            status="FAILED",

            start_time=task_start,

            end_time=task_end,

            duration_seconds=duration,

            error_message=(
                error_text
                + "\n"
                + full_traceback
            ),

            notebook_output=None
        )


        return "FAILED"


# ============================================================
# 18. WORKFLOW EXECUTION
# ============================================================

workflow_failed = False


for task in SELECTED_TASKS:


    # --------------------------------------------------------
    # 18.1 STOP AFTER FAILURE
    # --------------------------------------------------------

    if workflow_failed and STOP_ON_FAILURE:

        mark_task_skipped(

            task,

            "Skipped because STOP_ON_FAILURE is enabled "
            "and an earlier task failed."
        )

        continue


    # --------------------------------------------------------
    # 18.2 EXECUTE TASK
    # --------------------------------------------------------

    result = execute_workflow_task(
        task
    )


    if result == "FAILED":

        workflow_failed = True


# ============================================================
# 19. WORKFLOW SUMMARY
# ============================================================

workflow_end_time = utc_now()


workflow_duration = (
    workflow_end_time - workflow_start_time
).total_seconds()


success_count = sum(
    1
    for status in task_status.values()
    if status == "SUCCESS"
)


failed_count = sum(
    1
    for status in task_status.values()
    if status == "FAILED"
)


skipped_count = sum(
    1
    for status in task_status.values()
    if status == "SKIPPED"
)


summary_status = (
    "FAILED"
    if failed_count > 0
    else "SUCCESS"
)


# ============================================================
# 20. ADD PIPELINE SUMMARY REPORT
# ============================================================

summary_error_message = (

    f"dataset={DATASET_NAME}; "

    f"execution_mode={EXECUTION_MODE}; "

    f"selected={len(SELECTED_TASKS)}; "

    f"success={success_count}; "

    f"failed={failed_count}; "

    f"skipped={skipped_count}"
)


add_report_row(

    task_name="pipeline_execution_summary",

    notebook_path=WORKSPACE_ROOT,

    task_layer="PIPELINE",

    status=summary_status,

    start_time=workflow_start_time,

    end_time=workflow_end_time,

    duration_seconds=workflow_duration,

    error_message=summary_error_message,

    notebook_output=None
)


# ============================================================
# 21. SAVE EXECUTION REPORT
# ============================================================

if report_rows:

    report_df = spark.createDataFrame(

        report_rows,

        schema=REPORT_SCHEMA
    )


    # Ensure report DataFrame uses the exact table column order.

    report_df = report_df.select(
        *REPORT_SCHEMA.fieldNames()
    )


    (
        report_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(REPORT_TABLE)
    )


    print("\n" + "=" * 90)
    print("EXECUTION REPORT SAVED")
    print("=" * 90)

    print(f"Report table: {REPORT_TABLE}")
    print(f"Rows saved  : {len(report_rows)}")


# ============================================================
# 22. DISPLAY CURRENT RUN REPORT
# ============================================================

print("\n" + "=" * 90)
print("WORKFLOW EXECUTION SUMMARY")
print("=" * 90)

print(f"Pipeline name      : {PIPELINE_NAME}")
print(f"Run ID             : {RUN_ID}")
print(f"Dataset            : {DATASET_NAME}")
print(f"Pipeline layer     : {PIPELINE_LAYER}")
print(f"Execution mode     : {EXECUTION_MODE}")
print(f"Selected tasks     : {len(SELECTED_TASKS)}")
print(f"Successful tasks   : {success_count}")
print(f"Failed tasks       : {failed_count}")
print(f"Skipped tasks      : {skipped_count}")
print(f"Duration           : {workflow_duration:.2f} seconds")
print(f"Final status       : {summary_status}")

print("=" * 90)


# ============================================================
# 23. DISPLAY REPORT TABLE FOR CURRENT RUN
# ============================================================

current_run_report = (

    spark.table(REPORT_TABLE)

    .filter(
        F.col("run_id") == RUN_ID
    )

    .orderBy(
        "start_timestamp",
        "task_name"
    )

)


display(current_run_report)


# ============================================================
# 24. FINAL WORKFLOW STATUS
# ============================================================

if failed_count > 0:

    raise RuntimeError(

        f"Workflow failed. "

        f"Successful={success_count}, "

        f"Failed={failed_count}, "

        f"Skipped={skipped_count}. "

        f"Check {REPORT_TABLE} for task-level details."
    )


print("\n" + "=" * 90)
print("WORKFLOW COMPLETED SUCCESSFULLY")
print("=" * 90)

In [0]:
%sql
SELECT
    task_name,
    execution_status,
    duration_seconds,
    error_message,
    start_timestamp,
    end_timestamp
FROM workspace.ecommerce_gold.workflow_execution_report
WHERE run_id = 'a136684b-aa4a-4dc8-ab9f-6c0735c5214b'
ORDER BY start_timestamp;


## Verify workflow executions:


### View latest workflow executions

In [0]:
%sql

SELECT
    run_id,
    task_name,
    pipeline_layer,
    execution_status,
    duration_seconds,
    start_timestamp,
    end_timestamp
FROM workspace.ecommerce_gold.workflow_execution_report
ORDER BY start_timestamp DESC;


### Calculate task success and failure counts

In [0]:
%sql

SELECT
    task_name,
    COUNT(*) AS total_executions,
    SUM(
        CASE
            WHEN execution_status = 'SUCCESS'
            THEN 1 ELSE 0
        END
    ) AS successful_executions,
    SUM(
        CASE
            WHEN execution_status = 'FAILED'
            THEN 1 ELSE 0
        END
    ) AS failed_executions
FROM workspace.ecommerce_gold.workflow_execution_report
GROUP BY task_name
ORDER BY task_name;


### Find failed workflow runs

In [0]:
%sql

SELECT
    run_id,
    task_name,
    notebook_path,
    error_message,
    start_timestamp
FROM workspace.ecommerce_gold.workflow_execution_report
WHERE execution_status = 'FAILED'
ORDER BY start_timestamp DESC;